In [9]:
import numpy as np
import pyvista as pv
from scipy.spatial import ConvexHull

In [ ]:

def slice_polygon_to_layers(polygon, layer_thickness, bounds=None):
    
    # Получаем границы полигона
    if bounds is None:
        bounds = polygon.bounds
    min_z, max_z = bounds[4], bounds[5]
    
    # Генерируем уровни срезов
    z_levels = np.arange(min_z, max_z + layer_thickness, layer_thickness)
    
    slices_coords = []
    
    for z in z_levels:
        # Создаем плоскость для среза
        slice_plane = pv.Plane(
            center=(0, 0, z),
            direction=(0, 0, 1),
            i_size=10,  # Достаточно большой размер
            j_size=10
        )
        
        # Выполняем срез
        slice_result = polygon.slice(normal='z', origin=(0, 0, z))
        
       
        if slice_result.n_points > 0:
            # Получаем координаты вершин среза
            slice_points = slice_result.points
            
            min_point = np.min(slice_points, axis=0)
            
            # Нормализуем координаты так, чтобы минимальная точка была в нуле 
            normalized_points = slice_points - min_point
            
            slices_coords.append(normalized_points)
    
    return slices_coords


In [ ]:
# Пример использования
def create_example_polygon():
    """Создает пример 3D полигона для демонстрации"""
    # Создаем сферу как пример 3D объекта
    sphere = pv.Sphere(radius=1.0, theta_resolution=30, phi_resolution=30)
    return sphere


# Основной пример использования
if __name__ == "__main__":
    # Создаем пример полигона
    polygon = create_example_polygon()
    
    # Задаем толщину слэба
    layer_thickness = 0.2
    
    print(f"Границы полигона: {polygon.bounds}")
    print(f"Количество точек: {polygon.n_points}")
    
    # Выполняем разбиение на слэбы
    slices = slice_polygon_to_layers(polygon, layer_thickness)
    
    
    print(f"\nПолучено слэбов: {len(slices)}")
    
    # Выводим информацию о каждом слое
    for i, slice_coords in enumerate(slices):
        print(f"Слэб {i+1}:")
        print(f"  Количество точек: {len(slice_coords)}")
        print(f"  Границы: {np.min(slice_coords, axis=0)} - {np.max(slice_coords, axis=0)}")
    
    

Границы полигона: BoundsTuple(x_min = -0.9985334277153015,
            x_max =  0.9985334277153015,
            y_min = -0.9930633306503296,
            y_max =  0.9930633306503296,
            z_min = -1.0,
            z_max =  1.0)
Количество точек: 842

Получено слоев: 10
Слой 1:
  Количество точек: 60
  Границы: [0. 0. 0.] - [1.1988238 1.1922566 0.       ]
Слой 2:
  Количество точек: 60
  Границы: [0. 0. 0.] - [1.5963683 1.5876232 0.       ]
Слой 3:
  Количество точек: 60
  Границы: [0. 0. 0.] - [1.8303223 1.8202956 0.       ]
Слой 4:
  Количество точек: 60
  Границы: [0. 0. 0.] - [1.9568282 1.9461085 0.       ]
Слой 5:
  Количество точек: 60
  Границы: [0. 0. 0.] - [1.9970669 1.9861267 0.       ]
Слой 6:
  Количество точек: 60
  Границы: [0. 0. 0.] - [1.9568282 1.9461085 0.       ]
Слой 7:
  Количество точек: 60
  Границы: [0. 0. 0.] - [1.8303223 1.8202956 0.       ]
Слой 8:
  Количество точек: 60
  Границы: [0. 0. 0.] - [1.5963683 1.5876232 0.       ]
Слой 9:
  Количество точек: 

In [ ]:
def usable_area(polygon):
    # Проецируем все точки полигона на XY-плоскость
    projected_points = polygon.points.copy()
    projected_points[:, 2] = 0  # Обнуляем Z-координату
    
    hull = ConvexHull(projected_points[:, :2])  # Используем только X и Y координаты
   # Тут есть спергениальная идея но пока я нее знаю как ее реализовать
    return 